# Task 2: Connectivity vs Distance

Analyze connection probability as a function of cell-body distance for 66 source neurons in the MICrONS dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
connections_df = pd.read_feather("data/task_connections_1507.feather")
locations_df = pd.read_feather("data/task_cell_body_locations_1507.feather")

connections_df.head()

In [ ]:
print(len(set(connections_df["pre_pt_root_id"]) & set(connections_df["post_pt_root_id"])))

In [ ]:
pre_coords = locations_df.set_index("pt_root_id")[["pt_position_x", "pt_position_y", "pt_position_z"]]
post_coords = pre_coords

connections_with_length_df = connections_df.copy()
connections_with_length_df["distance_um"] = (
    np.linalg.norm(
        pre_coords.loc[connections_df["pre_pt_root_id"]].to_numpy()
        - post_coords.loc[connections_df["post_pt_root_id"]].to_numpy(),
        axis=1,
    )
    / 1000
)

connections_with_length_df.head()

In [ ]:
coords = locations_df.set_index("pt_root_id")[["pt_position_x", "pt_position_y", "pt_position_z"]]
pre_ids = connections_df["pre_pt_root_id"].unique()

pre_xyz = coords.loc[pre_ids].to_numpy()
all_xyz = coords.to_numpy()

potential_dist_um = np.linalg.norm(pre_xyz[:, None, :] - all_xyz[None, :, :], axis=2) / 1000

is_self = coords.index.to_numpy()[None, :] == pre_ids[:, None]
potential_dist_um = potential_dist_um[~is_self]

print(potential_dist_um.shape)

In [ ]:
bin_edges = np.arange(0, 850, 50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

n_potential, _ = np.histogram(potential_dist_um, bins=bin_edges)
n_connected, _ = np.histogram(connections_with_length_df["distance_um"], bins=bin_edges)
connection_prob = n_connected / n_potential

plt.figure(figsize=(8, 5))
plt.plot(bin_centers, connection_prob, "o-")
plt.xlabel("Cell-body distance (µm)")
plt.ylabel("Connection probability")
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

## Findings

Connection probability is highest at short cell-body distances (~12% within 50 µm) and decreases sharply with distance, falling to near zero beyond ~500 µm. This supports the hypothesis that synaptic connectivity becomes less likely as the distance between cell bodies increases.

**Limitations:** Only outgoing connections from 66 traced source neurons are analyzed; distance is measured between cell bodies, not along axons; and probabilities at very long distances are based on fewer connected pairs.